# Лабораторная работа 1. Линейная регрессия: RiskScore
Разделы: 1) EDA (отдельный ноутбук) · 2) нормализация и метрики · 3) класс линейной регрессии (analytic / GD / SGD, L1 / L2 / L1+L2) · 4) кросс-валидация и итоговая модель.

# Лабораторная работа 1. Часть 1: нормализация и метрики
Все алгоритмы реализованы вручную на NumPy; sklearn используется только для проверки.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)

DATA_DIR = "data"   # папка с train.csv и test.csv

: 

## Подготовка данных
Убираем полностью пустые строки, аномальные значения цели (±1e7), точные дубли и строки с пропусками.
Дубли удаляем до любых разбиений, иначе одинаковые строки попадут и в обучение, и в проверку.

In [ ]:
raw = pd.read_csv(f"{DATA_DIR}/train.csv")
ok = raw["RiskScore"].notna() & (raw["RiskScore"].abs() < 1e6)
data = raw[ok].drop_duplicates().dropna().reset_index(drop=True)
print(f"было {len(raw)}, осталось {len(data)}")

num_cols = [c for c in data.select_dtypes("number").columns if c != "RiskScore"]
X_demo = data[num_cols].to_numpy(dtype=float)
y_demo = data["RiskScore"].to_numpy(dtype=float)
X_demo.shape

## 1. Нормализация
* **z-score:** `z = (x − μ) / σ` (σ с делителем n, как в sklearn);
* **min-max:** `x' = (x − min) / (max − min)`.

Параметры считаются только по обучающей выборке (`fit`), а затем применяются к любым данным (`transform`).
Для константного столбца знаменатель заменяется на 1, чтобы не делить на ноль.

In [ ]:
class ZScore:
    """Стандартизация: (x - mean) / std."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        std = X.std(axis=0)                      # ddof=0
        self.std_ = np.where(std == 0, 1.0, std)
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return np.asarray(Z, dtype=float) * self.std_ + self.mean_


class MinMax:
    """Приведение к диапазону [0, 1]: (x - min) / (max - min)."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        span = X.max(axis=0) - self.min_
        self.span_ = np.where(span == 0, 1.0, span)
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.min_) / self.span_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return np.asarray(Z, dtype=float) * self.span_ + self.min_

In [ ]:
# проверка: обучаем на 80% строк, применяем к оставшимся 20% и сравниваем со sklearn
rng = np.random.RandomState(0)
perm = rng.permutation(len(X_demo))
tr, va = perm[:int(0.8 * len(perm))], perm[int(0.8 * len(perm)):]

for own, ref in [(ZScore(), StandardScaler()), (MinMax(), MinMaxScaler())]:
    own.fit(X_demo[tr]); ref.fit(X_demo[tr])
    d_tr = np.abs(own.transform(X_demo[tr]) - ref.transform(X_demo[tr])).max()
    d_va = np.abs(own.transform(X_demo[va]) - ref.transform(X_demo[va])).max()
    back = np.abs(own.inverse_transform(own.transform(X_demo[va])) - X_demo[va]).max()
    print(f"{type(own).__name__:7s} | макс. разница со sklearn: train {d_tr:.2e}, valid {d_va:.2e} | обратное преобразование: {back:.2e}")
    assert d_tr < 1e-8 and d_va < 1e-8

z = ZScore().fit_transform(X_demo[tr])
m = MinMax().fit_transform(X_demo[tr])
print("\nz-score: среднее ~", np.abs(z.mean(0)).max().round(12), "| std ~", z.std(0).round(6).min(), "-", z.std(0).round(6).max())
print("min-max: min =", m.min(), "| max =", m.max())

# константный столбец не ломает преобразование
const = np.column_stack([np.ones(5), np.arange(5.0)])
print("\nконстантный столбец:\n", ZScore().fit_transform(const).T, "\n", MinMax().fit_transform(const).T)

**Вывод.** Оба преобразования полностью совпадают со sklearn (разница на уровне 1e-16).
z-score нужен для градиентного спуска и регуляризации (признаки с разным масштабом иначе будут штрафоваться неодинаково),
min-max удобен, когда нужны значения в фиксированном диапазоне. Параметры на тестовой выборке не пересчитываются.

## 2. Метрики
Пусть `e_i = y_i − ŷ_i`.

* **MSE** = mean(e²) — сильнее штрафует большие ошибки; основная метрика соревнования.
* **MAE** = mean(|e|) — в единицах `RiskScore`.
* **R²** = 1 − Σe² / Σ(y − ȳ)² — доля объяснённой дисперсии.
* **MAPE** = mean(|e| / max(|y|, ε)), как в sklearn; возвращается доля (×100 даёт проценты).

In [ ]:
def _prep(y, pred):
    y = np.asarray(y, dtype=float).ravel()
    pred = np.asarray(pred, dtype=float).ravel()
    if y.shape != pred.shape:
        raise ValueError("y и pred должны иметь одинаковую длину")
    return y, pred


def mse(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean((y - pred) ** 2))


def mae(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean(np.abs(y - pred)))


def r2(y, pred):
    y, pred = _prep(y, pred)
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    if ss_tot == 0:                       # константная цель: как в sklearn
        return 1.0 if ss_res == 0 else 0.0
    return float(1.0 - ss_res / ss_tot)


def mape(y, pred):
    y, pred = _prep(y, pred)
    return float(np.mean(np.abs(y - pred) / np.maximum(np.abs(y), np.finfo(np.float64).eps)))

In [ ]:
# предсказания реальной модели (sklearn OLS) на отложенной части + искусственные случаи
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(ZScore().fit_transform(X_demo[tr]), y_demo[tr])
zs = ZScore().fit(X_demo[tr])
pred_real = model.predict(zs.transform(X_demo[va]))

cases = {
    "реальная модель": (y_demo[va], pred_real),
    "случайный шум": (rng.normal(50, 15, 1000), rng.normal(50, 15, 1000)),
    "идеальный прогноз": (y_demo[va], y_demo[va].copy()),
    "прогноз = среднее": (y_demo[va], np.full(len(va), y_demo[va].mean())),
}
rows = []
for name, (yt, yp) in cases.items():
    for metric, own, ref in [("MSE", mse, mean_squared_error), ("MAE", mae, mean_absolute_error),
                             ("R²", r2, r2_score), ("MAPE", mape, mean_absolute_percentage_error)]:
        a, b = own(yt, yp), ref(yt, yp)
        assert np.isclose(a, b, rtol=1e-10, atol=1e-12), (name, metric, a, b)
        rows.append((name, metric, a, b, abs(a - b)))
report = pd.DataFrame(rows, columns=["случай", "метрика", "своя", "sklearn", "|разница|"])
report.pivot(index="случай", columns="метрика", values="своя").round(5)

In [ ]:
report.round(8)

In [ ]:
# граничные случаи
print("R², константная цель, идеально:", r2([3, 3, 3], [3, 3, 3]), "| sklearn:", r2_score([3, 3, 3], [3, 3, 3]))
print("R², константная цель, с ошибкой:", r2([3, 3, 3], [2, 3, 4]), "| sklearn:", r2_score([3, 3, 3], [2, 3, 4]))
print("MAPE при y = 0:", mape([0.0, 1.0], [1.0, 1.0]), "| sklearn:", mean_absolute_percentage_error([0.0, 1.0], [1.0, 1.0]))

**Вывод.** Все четыре метрики совпадают со sklearn с точностью до ошибок округления на реальных и искусственных данных,
включая граничные случаи. Для итоговой оценки используется MSE, остальные метрики — для самопроверки.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
import matplotlib.pyplot as plt

## 1. Модель
Модель: `ŷ = w₀ + wᵀx`. Минимизируется функция

`J(w) = 1/(2n) · Σ (yᵢ − ŷᵢ)²  +  α · [ ρ·‖w‖₁ + (1−ρ)/2 · ‖w‖₂² ]`,

где `α` — сила регуляризации, а `ρ` задаёт тип штрафа: `ρ = 0` — L2 (Ridge), `ρ = 1` — L1 (Lasso),
`0 < ρ < 1` — L1+L2 (ElasticNet). Свободный член `w₀` не штрафуется. Такая запись совпадает с
параметризацией sklearn (`Lasso`, `ElasticNet`), а `Ridge(alpha=n·α)` соответствует `ρ = 0`.

**Способы обучения**
* `analytic` — нормальное уравнение: `(XᵀX/n + α(1−ρ)I) w = Xᵀy/n` на центрированных данных
  (опционально с весами объектов `sample_weight`)
  (решается через `np.linalg.solve`; если матрица вырождена — решение минимальной нормы через `np.linalg.lstsq`,
  как делает sklearn).
  Работает без регуляризации и для L2; L1 аналитически не решается (негладкий штраф).
* `gd` — градиентный спуск по всей выборке. Для L1-части используется проксимальный шаг
  (мягкое пороговое преобразование), что даёт точные нули в коэффициентах.
* `sgd` — стохастический градиентный спуск: обновление по одному объекту (или мини-батчу)
  в случайном порядке, шаг убывает со временем.

Данные для `gd` и `sgd` нужно стандартизировать — иначе сходимость будет медленной или расходящейся.

In [ ]:
class MyLinearRegression:
    def __init__(self, method="analytic", penalty=None, alpha=0.0, l1_ratio=0.5,
                 lr=None, max_iter=20000, tol=1e-10, batch_size=1,
                 lr0=0.01, lr_decay=0.05, random_state=0):
        self.method = method              # "analytic" | "gd" | "sgd"
        self.penalty = penalty            # None | "l1" | "l2" | "elasticnet"
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.lr = lr                      # шаг GD (по умолчанию 1/L)
        self.max_iter = max_iter          # GD: число итераций; SGD: число эпох
        self.tol = tol
        self.batch_size = batch_size      # SGD: размер мини-батча (1 — классический SGD)
        self.lr0 = lr0                    # SGD: начальный шаг
        self.lr_decay = lr_decay          # SGD: шаг на эпохе e равен lr0 / (1 + lr_decay * e)
        self.random_state = random_state

    # --- коэффициенты штрафа: J = MSE/2 + l1 * |w|_1 + l2/2 * |w|_2^2 ---
    def _penalties(self):
        if self.penalty is None or self.alpha == 0:
            return 0.0, 0.0
        if self.penalty == "l1":
            rho = 1.0
        elif self.penalty == "l2":
            rho = 0.0
        elif self.penalty == "elasticnet":
            rho = self.l1_ratio
        else:
            raise ValueError("penalty должен быть None, 'l1', 'l2' или 'elasticnet'")
        return self.alpha * rho, self.alpha * (1 - rho)

    def _loss(self, X, y, w, b):
        l1, l2 = self._penalties()
        r = X @ w + b - y
        return 0.5 * np.mean(r ** 2) + l1 * np.abs(w).sum() + 0.5 * l2 * (w @ w)

    @staticmethod
    def _soft(v, t):
        return np.sign(v) * np.maximum(np.abs(v) - t, 0.0)

    def fit(self, X, y, sample_weight=None):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        n, d = X.shape
        l1, l2 = self._penalties()
        self.loss_history_ = []
        if sample_weight is not None and self.method != "analytic":
            raise ValueError("sample_weight поддерживается только для method='analytic'")

        if self.method == "analytic":
            if l1 > 0:
                raise ValueError("analytic не поддерживает L1; используйте method='gd' или 'sgd'")
            # веса объектов (по умолчанию все равны 1): минимизируется взвешенная сумма квадратов
            w_obj = np.ones(n) if sample_weight is None else np.asarray(sample_weight, dtype=float)
            w_obj = w_obj / w_obj.sum()
            x_mean, y_mean = w_obj @ X, w_obj @ y
            Xc, yc = X - x_mean, y - y_mean
            A = Xc.T @ (Xc * w_obj[:, None]) + l2 * np.eye(d)
            rhs = Xc.T @ (yc * w_obj)
            if np.linalg.matrix_rank(A) == d:
                w = np.linalg.solve(A, rhs)                      # обычный случай: матрица обратима
            else:
                w = np.linalg.lstsq(A, rhs, rcond=None)[0]       # вырожденный случай: решение минимальной нормы
            self.coef_, self.intercept_ = w, y_mean - x_mean @ w

        elif self.method == "gd":
            x_mean, y_mean = X.mean(axis=0), y.mean()
            Xc, yc = X - x_mean, y - y_mean            # центрирование: свободный член находится отдельно
            G = Xc.T @ Xc / n
            step = self.lr if self.lr else 1.0 / (np.linalg.eigvalsh(G)[-1] + l2)   # 1/L гарантирует сходимость
            c = Xc.T @ yc / n
            w = np.zeros(d)
            prev = np.inf
            for it in range(self.max_iter):
                grad = G @ w - c + l2 * w                # градиент гладкой части
                w = self._soft(w - step * grad, step * l1)   # проксимальный шаг для L1
                if it % 10 == 0:
                    cur = self._loss(Xc, yc, w, 0.0)
                    self.loss_history_.append((it, cur))
                    if abs(prev - cur) < self.tol:
                        break
                    prev = cur
            self.n_iter_ = it + 1
            self.coef_, self.intercept_ = w, y_mean - x_mean @ w

        elif self.method == "sgd":
            rng = np.random.RandomState(self.random_state)
            w, b = np.zeros(d), y.mean()
            bs = self.batch_size
            for epoch in range(self.max_iter):
                eta = self.lr0 / (1.0 + self.lr_decay * epoch)
                order = rng.permutation(n)
                for s in range(0, n, bs):
                    idx = order[s:s + bs]
                    r = X[idx] @ w + b - y[idx]
                    w = w - eta * (X[idx].T @ r / len(idx) + l2 * w)
                    b = b - eta * r.mean()
                    if l1 > 0:
                        w = self._soft(w, eta * l1)
                self.loss_history_.append((epoch, self._loss(X, y, w, b)))
            self.n_iter_ = self.max_iter
            self.coef_, self.intercept_ = w, b
        else:
            raise ValueError("method должен быть 'analytic', 'gd' или 'sgd'")
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

## 2. Данные
Признаки: все числовые + one-hot для категориальных (без даты). Делим 80/20 случайно,
нормализуем z-score по обучающей части.

In [ ]:
cat_cols = ["MaritalStatus", "HomeOwnershipStatus", "LoanPurpose", "EmploymentStatus", "EducationLevel"]
X_df = pd.get_dummies(data.drop(columns=["RiskScore", "ApplicationDate"]), columns=cat_cols,
                      drop_first=True, dtype=float)
y_all = data["RiskScore"].to_numpy(float)
X_all = X_df.to_numpy(float)

perm = np.random.RandomState(1).permutation(len(y_all))
cut = int(0.8 * len(perm))
tr, va = perm[:cut], perm[cut:]
scaler = ZScore().fit(X_all[tr])
Xtr, Xva = scaler.transform(X_all[tr]), scaler.transform(X_all[va])
ytr, yva = y_all[tr], y_all[va]
print("признаков:", Xtr.shape[1], "| train:", len(ytr), "| valid:", len(yva))

rank = np.linalg.matrix_rank(Xtr)
print(f"ранг матрицы признаков: {rank} из {Xtr.shape[1]}")
# какая именно линейная зависимость между признаками
_, _, vt = np.linalg.svd(Xtr, full_matrices=False)
load = pd.Series(vt[-1], index=X_df.columns)
print("признаки точной зависимости (веса в нулевом направлении):")
print(load[load.abs() > 0.05].sort_values().round(3).to_string())

## 3. Без регуляризации: аналитика, GD, SGD против sklearn

In [ ]:
def report(models, Xt, yt, Xv, yv, ref_name):
    ref_coef = models[ref_name].coef_
    rows = []
    for name, m in models.items():
        p = m.predict(Xv)
        rows.append({"модель": name, "MSE": mse(yv, p), "MAE": mae(yv, p), "R²": r2(yv, p),
                     "макс|Δ коэф|": np.max(np.abs(np.ravel(m.coef_) - np.ravel(ref_coef))),
                     "нулей": int(np.sum(np.abs(np.ravel(m.coef_)) < 1e-10))})
    return pd.DataFrame(rows).set_index("модель")

models = {
    "sklearn LinearRegression": LinearRegression().fit(Xtr, ytr),
    "своя: analytic": MyLinearRegression("analytic").fit(Xtr, ytr),
    "своя: GD": MyLinearRegression("gd", max_iter=200000).fit(Xtr, ytr),
    "своя: SGD": MyLinearRegression("sgd", max_iter=60, lr0=0.01, lr_decay=0.1).fit(Xtr, ytr),
    "sklearn SGDRegressor": SGDRegressor(penalty=None, learning_rate="invscaling", eta0=0.01,
                                         max_iter=60, tol=None, random_state=0).fit(Xtr, ytr),
}
print("итераций GD:", models["своя: GD"].n_iter_)
report(models, Xtr, ytr, Xva, yva, "sklearn LinearRegression").round(6)

**Замечание про вырожденность.** Матрица признаков имеет ранг на единицу меньше числа столбцов: `BaseInterestRate`
оказывается точной линейной комбинацией `CreditScore`, `LoanAmount` и `LoanDuration`. Поэтому коэффициенты OLS
определены неоднозначно (отсюда большой «макс|Δ коэф|» у аналитики), но **предсказания** и метрики совпадают.
В таких случаях `fit` берёт решение минимальной нормы, а регуляризация (Ridge/Lasso) делает решение единственным.
Аналитическое решение отличается от sklearn только выбором решения среди эквивалентных.

**Вывод.** Аналитическое решение совпадает с `LinearRegression` до ошибок округления. GD сходится к тому же
решению, но медленно: из-за сильной коррелированности признаков матрица `XᵀX` плохо обусловлена.
SGD немного уступает по MSE (≈ на 0.9) — так же, как и `SGDRegressor` из sklearn: на плохо обусловленной задаче
шаги со случайным порядком объектов не доходят до точного оптимума, а коэффициенты «шумят».
У аналитики, GD и sklearn метрики на валидации практически совпадают.

In [ ]:
# сходимость GD и SGD
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
it, loss = zip(*models["своя: GD"].loss_history_)
ax[0].plot(it, loss); ax[0].set(xscale="log", yscale="log", xlabel="итерация", ylabel="функция потерь", title="Градиентный спуск")
ep, loss = zip(*models["своя: SGD"].loss_history_)
ax[1].plot(ep, loss); ax[1].set(yscale="log", xlabel="эпоха", ylabel="функция потерь", title="Стохастический градиентный спуск")
for a in ax: a.axhline(0.5 * mse(ytr, models["своя: analytic"].predict(Xtr)), color="r", ls="--", label="оптимум (analytic)"); a.legend()
plt.tight_layout(); plt.show()

## 4. Регуляризация: L2, L1, L1+L2
Сравниваем с `Ridge`, `Lasso`, `ElasticNet` из sklearn при одинаковом `α` (для Ridge — `n·α`, см. формулу выше).

In [ ]:
n_tr = len(ytr)
alpha_l2, alpha_l1, alpha_en = 0.05, 0.05, 0.05

ridge = {
    "sklearn Ridge": Ridge(alpha=n_tr * alpha_l2).fit(Xtr, ytr),
    "своя: L2 analytic": MyLinearRegression("analytic", "l2", alpha_l2).fit(Xtr, ytr),
    "своя: L2 GD": MyLinearRegression("gd", "l2", alpha_l2, max_iter=200000).fit(Xtr, ytr),
    "своя: L2 SGD": MyLinearRegression("sgd", "l2", alpha_l2, max_iter=60, lr0=0.01, lr_decay=0.1).fit(Xtr, ytr),
}
print("L2 (Ridge), α =", alpha_l2)
report(ridge, Xtr, ytr, Xva, yva, "sklearn Ridge").round(6)

In [ ]:
lasso = {
    "sklearn Lasso": Lasso(alpha=alpha_l1, tol=1e-10, max_iter=100000).fit(Xtr, ytr),
    "своя: L1 GD": MyLinearRegression("gd", "l1", alpha_l1, max_iter=200000).fit(Xtr, ytr),
    "своя: L1 SGD": MyLinearRegression("sgd", "l1", alpha_l1, max_iter=60, lr0=0.01, lr_decay=0.1).fit(Xtr, ytr),
}
print("L1 (Lasso), α =", alpha_l1)
report(lasso, Xtr, ytr, Xva, yva, "sklearn Lasso").round(6)

In [ ]:
enet = {
    "sklearn ElasticNet": ElasticNet(alpha=alpha_en, l1_ratio=0.5, tol=1e-10, max_iter=100000).fit(Xtr, ytr),
    "своя: L1+L2 GD": MyLinearRegression("gd", "elasticnet", alpha_en, 0.5, max_iter=200000).fit(Xtr, ytr),
    "своя: L1+L2 SGD": MyLinearRegression("sgd", "elasticnet", alpha_en, 0.5, max_iter=60, lr0=0.01, lr_decay=0.1).fit(Xtr, ytr),
}
print("L1+L2 (ElasticNet), α =", alpha_en, ", ρ = 0.5")
report(enet, Xtr, ytr, Xva, yva, "sklearn ElasticNet").round(6)

**Вывод.** Для L2 аналитическое решение и GD совпадают с `Ridge`. Для L1 и L1+L2 градиентный спуск с проксимальным
шагом получает те же коэффициенты, что и `Lasso`/`ElasticNet`, причём часть коэффициентов обнуляется (отбор признаков).
SGD даёт близкий по качеству результат, но точных нулей почти нет — из-за шума случайных шагов коэффициенты колеблются вокруг нуля.

In [ ]:
# влияние силы регуляризации: качество и число ненулевых коэффициентов (GD)
rows = []
for a in [0.0, 0.001, 0.01, 0.05, 0.1, 0.3, 1.0]:
    for pen in ["l2", "l1"]:
        m = MyLinearRegression("gd", pen, a, max_iter=100000).fit(Xtr, ytr)
        rows.append({"α": a, "штраф": pen, "MSE valid": mse(yva, m.predict(Xva)),
                     "ненулевых коэф.": int(np.sum(np.abs(m.coef_) > 1e-8))})
pd.DataFrame(rows).pivot(index="α", columns="штраф").round(3)

**Вывод.** Небольшая регуляризация почти не меняет качество, но при росте `α` L1 быстро обнуляет признаки,
а ошибка растёт. Регуляризация нужна не для снижения MSE на этих данных, а для устойчивости коэффициентов
при мультиколлинеарности и для отбора признаков. Значение `α` подбирается кросс-валидацией (следующая часть).

# Часть 3: кросс-валидация и итоговая модель
## 1. Кросс-валидация k-fold и leave-one-out
* **k-fold:** индексы перемешиваются, делятся на `k` почти равных частей; каждая часть по очереди служит проверочной.
* **Leave-one-out (LOO):** частный случай `k = n`: проверочная часть — один объект.

Нормализация внутри каждого фолда обучается **только на train-части фолда** (иначе информация из проверочной части
«подтекает» в обучение).

In [ ]:
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def my_kfold(n, k=5, shuffle=True, seed=42):
    """Генератор пар (train_idx, valid_idx). Размеры частей отличаются не более чем на 1."""
    idx = np.arange(n)
    if shuffle:
        np.random.RandomState(seed).shuffle(idx)
    sizes = np.full(k, n // k)
    sizes[: n % k] += 1
    start = 0
    for size in sizes:
        valid = np.sort(idx[start:start + size])
        train = np.sort(np.concatenate([idx[:start], idx[start + size:]]))
        yield train, valid
        start += size


def my_loo(n):
    idx = np.arange(n)
    for i in range(n):
        yield np.delete(idx, i), idx[i:i + 1]


def my_cv_predict(make_model, X, y, splits, scaler_cls=ZScore):
    """Out-of-fold предсказания: каждый объект предсказывается моделью, которая его не видела."""
    X, y = np.asarray(X, float), np.asarray(y, float)
    pred = np.zeros(len(y))
    for tr_idx, va_idx in splits:
        scaler = scaler_cls().fit(X[tr_idx]) if scaler_cls else None
        Xt = scaler.transform(X[tr_idx]) if scaler else X[tr_idx]
        Xv = scaler.transform(X[va_idx]) if scaler else X[va_idx]
        pred[va_idx] = make_model().fit(Xt, y[tr_idx]).predict(Xv)
    return pred


def cv_scores(y, pred):
    return {"MSE": mse(y, pred), "MAE": mae(y, pred), "R²": r2(y, pred)}

### Проверка разбиений и результатов относительно sklearn

In [ ]:
n = len(y_all)
# 1) индексы k-fold совпадают с KFold из sklearn (при тех же shuffle и seed)
for (a_tr, a_va), (b_tr, b_va) in zip(my_kfold(n, 5, True, 42), KFold(5, shuffle=True, random_state=42).split(X_all)):
    assert np.array_equal(a_va, b_va) and np.array_equal(a_tr, b_tr)
print("k-fold: разбиения совпадают с sklearn (5 фолдов, перемешивание, seed=42)")

# 2) предсказания и MSE: своя OLS + свой z-score против Pipeline(StandardScaler, LinearRegression)
own_pred = my_cv_predict(lambda: MyLinearRegression("analytic"), X_all, y_all, my_kfold(n, 5, True, 42))
ref_pred = cross_val_predict(make_pipeline(StandardScaler(), LinearRegression()), X_all, y_all,
                             cv=KFold(5, shuffle=True, random_state=42))
print(f"5-fold MSE: своя = {mse(y_all, own_pred):.6f} | sklearn = {mean_squared_error(y_all, ref_pred):.6f} "
      f"| макс|Δ предсказаний| = {np.abs(own_pred - ref_pred).max():.2e}")

In [ ]:
# LOO: на подвыборке 600 объектов (полный перебор 9 332 обучений слишком долог)
m = 600
sub = np.random.RandomState(0).choice(n, m, replace=False)
Xs, ys = X_all[sub], y_all[sub]
for (a_tr, a_va), (b_tr, b_va) in zip(my_loo(m), LeaveOneOut().split(Xs)):
    assert np.array_equal(a_tr, b_tr) and np.array_equal(a_va, b_va)
own_loo = my_cv_predict(lambda: MyLinearRegression("analytic"), Xs, ys, my_loo(m))
ref_loo = cross_val_predict(make_pipeline(StandardScaler(), LinearRegression()), Xs, ys, cv=LeaveOneOut())
print(f"LOO (m={m}) MSE: своя = {mse(ys, own_loo):.6f} | sklearn = {mean_squared_error(ys, ref_loo):.6f} "
      f"| макс|Δ| = {np.abs(own_loo - ref_loo).max():.2e}")

**LOO для всей выборки.** Для МНК остатки LOO считаются без переобучения: `e_i^(LOO) = e_i / (1 − h_ii)`, где
`h_ii` — диагональ «шляпной» матрицы `H = X(XᵀX)⁺Xᵀ`. Проверим формулу на тех же 600 объектах и посчитаем на всех данных.

In [ ]:
def loo_ols_closed_form(X, y):
    """LOO-предсказания МНК по формуле через диагональ шляпной матрицы (без повторных обучений)."""
    Xa = np.column_stack([np.ones(len(y)), X])
    U, s, _ = np.linalg.svd(Xa, full_matrices=False)
    U = U[:, s > s.max() * 1e-10]                    # устойчиво к вырожденности признаков
    h = np.sum(U ** 2, axis=1)                       # diag(H)
    fitted = U @ (U.T @ y)
    return y - (y - fitted) / (1 - h)


print(f"LOO по формуле на тех же {m} объектах: {mse(ys, loo_ols_closed_form(Xs, ys)):.6f} "
      f"(перебором: {mse(ys, own_loo):.6f})")
print(f"LOO MSE на всех {n} объектах (МНК, формула): {mse(y_all, loo_ols_closed_form(X_all, y_all)):.4f}")

**Вывод.** Свои `my_kfold` и `my_loo` дают те же разбиения, что и sklearn, а out-of-fold предсказания и MSE
совпадают с `cross_val_predict` до ошибок округления. Простой МНК на исходных признаках даёт
CV-MSE около 50 — это «нулевая линия», с которой мы сравниваем всё дальнейшее.

### Подбор силы регуляризации кросс-валидацией

In [ ]:
rows = []
for a in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    p = my_cv_predict(lambda: MyLinearRegression("analytic", "l2", a), X_all, y_all, my_kfold(n, 5, True, 42))
    rows.append({"α (L2)": a, **cv_scores(y_all, p)})
pd.DataFrame(rows).set_index("α (L2)").round(4)

На исходных признаках регуляризация почти не влияет (MSE ≈ 50 при любом разумном `α`): проблема не в переобучении,
а в том, что модель слишком проста для этой зависимости. Поэтому дальше усложняем **признаки**, оставаясь в классе
линейных моделей (линейность по параметрам).

## 2. Поиск порогов линейной моделью (L1-регуляризация)
В EDA видно, что зависимости ступенчатые: риск меняется при «круглых» значениях признаков (кредитный рейтинг 600/650/…).
Чтобы найти такие пороги, не используя ничего, кроме линейной регрессии, строим для каждого признака «ступеньки»
`1[x ≥ t]` на мелкой сетке порогов и обучаем **свою** линейную регрессию с L1-штрафом (`MyLinearRegression`, GD).
L1 оставляет ненулевые коэффициенты только у нужных порогов.

In [ ]:
def grid(lo, hi, step):
    return list(np.round(np.arange(lo, hi + 1e-9, step), 6))

FINE_GRID = {
    "CreditScore": grid(310, 840, 10), "DebtToIncomeRatio": grid(0.05, 0.85, 0.025),
    "CreditCardUtilizationRate": grid(0.05, 0.9, 0.05), "LengthOfCreditHistory": grid(2, 29, 1),
    "AnnualIncome": grid(10_000, 150_000, 5_000) + grid(175_000, 400_000, 25_000),
    "NetWorth": [1_000 * 2 ** i for i in range(13)] + [5_000 * 1.5 ** i for i in range(12)],
    "TotalDebtToIncomeRatio": grid(0.05, 1, 0.05),
}
fine_names, fine_cols = [], []
for c, ts in FINE_GRID.items():
    for t in ts:
        fine_names.append(f"{c} ≥ {t:g}")
        fine_cols.append((data[c].to_numpy() >= t).astype(float))
F_fine = np.column_stack(fine_cols)
Z_fine = ZScore().fit_transform(np.hstack([X_all, F_fine]))
names_all = list(X_df.columns) + fine_names

l1_model = MyLinearRegression("gd", "l1", alpha=0.05, max_iter=40000, tol=1e-9).fit(Z_fine, y_all)
coef = pd.Series(l1_model.coef_, index=names_all)
steps_found = coef[[nm for nm in fine_names]]
steps_found = steps_found[steps_found != 0]
print(f"ненулевых коэффициентов: {int((coef != 0).sum())} из {len(coef)}, из них ступенек: {len(steps_found)}")
steps_found.sort_values(key=np.abs, ascending=False).head(25).round(2).to_frame("коэффициент")

Наибольшие коэффициенты — у «круглых» порогов: `CreditScore ≥ 600/650/700/750`, `AnnualIncome ≥ 30k/50k/80k/120k`,
`DebtToIncomeRatio ≥ 0.2/0.3/0.4/0.5`, ступеньки по `NetWorth` в районе 50k–500k и по `CreditCardUtilizationRate`,
`LengthOfCreditHistory`. Их мы и берём в словарь `STEP_THRESHOLDS` (значения округлены).

## 3. Итоговая модель
Лекция: «линейная регрессия линейна по параметрам, но необязательно линейна по признакам». Используем только
линейную регрессию (свой класс `MyLinearRegression`), никаких других моделей, в том числе для отбора признаков.

**Шаг 1. Признаки.**
* ступеньки на найденных порогах (раздел 2);
* логарифмы тяжёлых правых хвостов (`log1p`) и логарифмы финансовых отношений (долговая нагрузка, кредит к доходу и т. п.);
* кусочно-линейный базис по ключевым признакам: `x`, `max(x − k₁, 0)`, `max(x − k₂, 0)`, … с узлами в квантилях
  (обучается только на train-части);
* все остальные числовые признаки и one-hot категории.

**Шаг 2. Две линейные регрессии и вес между ними.** Распределение `RiskScore` двугорбое, а риск при росте дохода и
кредитного рейтинга «насыщается» (S-образная зависимость). Поэтому строим две линейные регрессии «низкий риск» и
«высокий риск» и итоговый прогноз `ŷ = (1 − w(x))·f_low(x) + w(x)·f_high(x)`.
* Начальное разбиение: по медиане `RiskScore`, затем несколько итераций: каждый объект относится к той ветке,
  чей прогноз ближе к его `RiskScore`; ветки переобучаются (L2-регуляризованный МНК).
* Вес `w(x)` — тоже линейная регрессия. Для объекта нужный вес `w* = (y − f_low)/(f_high − f_low)`, обрезанный в `[0, 1]`.
  Минимизируемая ошибка `Σ (y − f_low − w·(f_high − f_low))²` равна `Σ (f_high − f_low)²·(w* − w)²`, поэтому вес `w`
  обучается **взвешенным МНК** с весами `(f_high − f_low)²`.
* Линейный индекс `z = wᵀx` затем «калибруется» кусочно-линейной функцией от `z` (тоже линейная регрессия по hinge-признакам)
  и обрезается в `[0, 1]`. Это даёт плавный S-образный переход без логистической функции.

In [ ]:
CATEGORICAL = ["MaritalStatus", "HomeOwnershipStatus", "LoanPurpose", "EmploymentStatus", "EducationLevel"]
LOG_COLS = ["TotalDebtToIncomeRatio", "MonthlyLoanPayment", "TotalAssets", "NetWorth", "TotalLiabilities",
            "SavingsAccountBalance", "CheckingAccountBalance", "LoanAmount", "MonthlyDebtPayments",
            "AnnualIncome", "MonthlyIncome"]
STEP_THRESHOLDS = {            # пороги, найденные L1-регрессией в разделе 2 (округлённые значения)
    "CreditScore": [600, 650, 700, 750],
    "AnnualIncome": [30_000, 50_000, 80_000, 120_000],
    "DebtToIncomeRatio": [0.2, 0.3, 0.4, 0.5],
    "NetWorth": [50_000, 100_000, 250_000, 500_000],
    "CreditCardUtilizationRate": [0.2, 0.4, 0.6],
    "LengthOfCreditHistory": [3, 7, 10],
}
KEY_COLS = ["CreditScore", "MonthlyIncome", "AnnualIncome", "BaseInterestRate", "InterestRate",
            "TotalDebtToIncomeRatio", "DebtToIncomeRatio", "NetWorth", "TotalAssets", "LoanAmount",
            "MonthlyLoanPayment", "Age", "LengthOfCreditHistory", "CreditCardUtilizationRate",
            "r_loan_inc", "r_loanpay_mi", "r_debtpay_mi", "r_nw_inc", "r_sav_inc", "r_liab_inc", "r_assets_inc"]


def make_features(df, columns=None):
    """Таблица признаков для обучения/теста. `columns` — порядок столбцов, зафиксированный на train."""
    X = df.drop(columns=["RiskScore", "ApplicationDate", "ID"], errors="ignore")
    X = pd.get_dummies(X, columns=CATEGORICAL, drop_first=True, dtype=float)
    for c in LOG_COLS:
        X["log_" + c] = np.log1p(X[c])
    inc, mi = df["AnnualIncome"], df["MonthlyIncome"]
    ratios = pd.DataFrame({
        "loan_inc": df.LoanAmount / inc, "debtpay_mi": df.MonthlyDebtPayments / mi,
        "loanpay_mi": df.MonthlyLoanPayment / mi, "assets_inc": df.TotalAssets / inc,
        "nw_inc": df.NetWorth / inc, "liab_assets": df.TotalLiabilities / df.TotalAssets,
        "liab_inc": df.TotalLiabilities / inc, "sav_inc": df.SavingsAccountBalance / inc,
        "chk_inc": df.CheckingAccountBalance / inc,
        "liq_debt": (df.SavingsAccountBalance + df.CheckingAccountBalance) / (df.MonthlyDebtPayments + 1),
        "hist_age": df.LengthOfCreditHistory / df.Age, "exp_age": df.Experience / df.Age,
        "tenure_exp": df.JobTenure / (df.Experience + 1), "infl_rate": df.InterestRate / df.BaseInterestRate,
        "cs_inq": df.CreditScore / (df.NumberOfCreditInquiries + 1),
        "inc_dep": inc / (df.NumberOfDependents + 1),
        "util_x_loan": df.CreditCardUtilizationRate * df.LoanAmount / inc,
    })
    ratios = np.sign(ratios) * np.log1p(ratios.abs())          # знаковый логарифм сжимает хвосты
    for c in ratios:
        X["r_" + c] = ratios[c].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
    for c, thresholds in STEP_THRESHOLDS.items():               # ступеньки: 1, если значение достигло порога
        for t in thresholds:
            X[f"{c}>={t:g}"] = (df[c] >= t).astype(float)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X


feat_train = make_features(data)
FEATURE_COLUMNS = list(feat_train.columns)
KEY_IDX = [FEATURE_COLUMNS.index(c) for c in KEY_COLS]
X_feat = feat_train.to_numpy(float)
print("признаков в таблице:", X_feat.shape[1], "| ключевых для кусочно-линейного базиса:", len(KEY_IDX))

In [ ]:
class PiecewiseLinearBasis:
    """Кусочно-линейный базис по каждому столбцу: [x, max(x−k1,0), ..., max(x−k_m,0)], узлы — квантили на train."""

    def __init__(self, n_knots=6):
        self.n_knots = n_knots

    def fit(self, X):
        q = np.linspace(0, 1, self.n_knots)
        self.knots_ = [np.unique(np.quantile(X[:, j], q)) for j in range(X.shape[1])]
        return self

    def transform(self, X):
        cols = []
        for j, k in enumerate(self.knots_):
            x = np.clip(X[:, j], k[0], k[-1])                  # вне диапазона train значение «замораживается»
            cols.append(x)
            cols.extend(np.maximum(x - kk, 0.0) for kk in k[1:-1])
        return np.column_stack(cols)


def ridge(a, n_eff):
    """L2-регуляризованный аналитический МНК; `a` — сила штрафа в единицах Σ(ошибка²) + a·‖w‖²."""
    return MyLinearRegression("analytic", "l2", a / n_eff)


class BasisRidge:
    """Одна линейная регрессия на расширенных признаках (для сравнения с двухветочной моделью)."""

    def __init__(self, key_idx, n_knots=6, alpha=10.0):
        self.key_idx, self.n_knots, self.alpha = key_idx, n_knots, alpha

    def fit(self, X, y):
        self.basis_ = PiecewiseLinearBasis(self.n_knots).fit(X[:, self.key_idx])
        Z = np.hstack([X, self.basis_.transform(X[:, self.key_idx])])
        self.scaler_ = ZScore().fit(Z)
        self.model_ = ridge(self.alpha, len(y)).fit(self.scaler_.transform(Z), y)
        return self

    def predict(self, X):
        Z = np.hstack([X, self.basis_.transform(X[:, self.key_idx])])
        return self.model_.predict(self.scaler_.transform(Z))

In [ ]:
class TwoRegimeLinear:
    """ŷ = (1 − w(x))·f_low(x) + w(x)·f_high(x): три линейные регрессии (две ветки и вес между ними)."""

    def __init__(self, key_idx, n_knots=6, alpha_branch=10.0, alpha_gate=10.0, link_knots=8, em_iter=50):
        self.key_idx, self.n_knots = key_idx, n_knots
        self.alpha_branch, self.alpha_gate = alpha_branch, alpha_gate
        self.link_knots, self.em_iter = link_knots, em_iter

    def _design(self, X, fit=False):
        if fit:
            self.basis_ = PiecewiseLinearBasis(self.n_knots).fit(X[:, self.key_idx])
        Z = np.hstack([X, self.basis_.transform(X[:, self.key_idx])])
        if fit:
            self.scaler_ = ZScore().fit(Z)
        return self.scaler_.transform(Z)

    @staticmethod
    def _hinge(z, knots):
        z = np.clip(z, knots[0], knots[-1])
        return np.column_stack([z] + [np.maximum(z - k, 0.0) for k in knots[1:-1]])

    def _weight(self, A):
        z = self.index_.predict(A)
        return np.clip(self.link_.predict(self._hinge(z, self.link_knots_)), 0.0, 1.0)

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        A = self._design(X, fit=True)
        # 1) разметка веток: итеративное переназначение объектов ближайшей по прогнозу ветке
        upper = y > np.median(y)
        for _ in range(self.em_iter):
            low = ridge(self.alpha_branch, (~upper).sum()).fit(A[~upper], y[~upper])
            high = ridge(self.alpha_branch, upper.sum()).fit(A[upper], y[upper])
            new = np.abs(y - high.predict(A)) < np.abs(y - low.predict(A))
            if np.array_equal(new, upper):
                break
            upper = new
        self.low_, self.high_ = low, high
        # 2) вес ветки: взвешенный МНК по «идеальному» весу w*
        L, H = low.predict(A), high.predict(A)
        delta = H - L
        safe = np.where(np.abs(delta) < 1e-6, 1e-6, delta)
        w_star = np.clip((y - L) / safe, 0.0, 1.0)
        sw = delta ** 2
        self.index_ = ridge(self.alpha_gate, sw.sum()).fit(A, w_star, sample_weight=sw)
        z = self.index_.predict(A)
        # 3) кусочно-линейная калибровка индекса (линейная регрессия по hinge-признакам индекса)
        self.link_knots_ = np.unique(np.quantile(z, np.linspace(0, 1, self.link_knots)))
        self.link_ = ridge(1e-3, sw.sum()).fit(self._hinge(z, self.link_knots_), w_star, sample_weight=sw)
        return self

    def predict(self, X):
        A = self._design(np.asarray(X, float))
        w = self._weight(A)
        return (1 - w) * self.low_.predict(A) + w * self.high_.predict(A)

## 4. Кросс-валидация итоговой модели
Сравниваем несколько вариантов на одних и тех же 5 фолдах (`my_kfold`, seed = 0). Для моделей с собственным
масштабированием (`scaler_cls=None`) нормализация выполняется внутри `fit`.

In [ ]:
def row(name, make_model, X, scaler_cls=None):
    p = my_cv_predict(make_model, X, y_all, my_kfold(n, 5, True, 0), scaler_cls)
    return {"модель": name, **cv_scores(y_all, p), "MAPE": mape(y_all, p)}

raw_idx = [FEATURE_COLUMNS.index(c) for c in X_df.columns if c in FEATURE_COLUMNS]
rows = [
    row("1. МНК на исходных признаках", lambda: MyLinearRegression("analytic"), X_feat[:, raw_idx], ZScore),
    row("2. МНК + ступеньки, логарифмы, отношения", lambda: MyLinearRegression("analytic", "l2", 1e-3), X_feat, ZScore),
    row("3. + кусочно-линейный базис", lambda: BasisRidge(KEY_IDX), X_feat),
    row("4. две ветки + вес (итоговая)", lambda: TwoRegimeLinear(KEY_IDX), X_feat),
]
cv_table = pd.DataFrame(rows).set_index("модель")
cv_table.round(4)

**Вывод.** Каждое усложнение признаков снижает ошибку: ступеньки на найденных порогах, логарифмы и отношения убирают
влияние тяжёлых хвостов и ловят ступенчатый характер зависимости, кусочно-линейный базис описывает насыщение, а две
ветки с плавным весом — двугорбое распределение цели. Все этапы — это линейные регрессии (обучаются своим классом
`MyLinearRegression`). Итоговая CV-ошибка ≈ 21; запас до порога 18.5 небольшой, поэтому требуется проверка на Kaggle.

## 5. Предсказание для test.csv и файл решения

In [ ]:
test = pd.read_csv(f"{DATA_DIR}/test.csv")
X_test = make_features(test, FEATURE_COLUMNS).to_numpy(float)
assert np.isfinite(X_test).all() and X_test.shape[1] == X_feat.shape[1]

final_model = TwoRegimeLinear(KEY_IDX).fit(X_feat, y_all)
pred_test = final_model.predict(X_test)

submission = pd.DataFrame({"ID": test["ID"], "RiskScore": pred_test})
assert len(submission) == len(test) == 5000 and submission["ID"].is_unique
assert np.isfinite(submission["RiskScore"]).all()
submission.to_csv("submission.csv", index=False)
print("train RiskScore: ", y_all.min().round(1), "-", y_all.max().round(1), "| среднее", y_all.mean().round(2))
print("test прогноз:    ", pred_test.min().round(1), "-", pred_test.max().round(1), "| среднее", pred_test.mean().round(2))
submission.head()

Файл `submission.csv` (колонки `ID,RiskScore`) загружается на Kaggle. Качество на Kaggle ожидается чуть хуже, чем
CV на train, поскольку гиперпараметры подбирались на train.